# 🔤 Day 04: String DP — Subsequences, Edits & Palindromes

---

## The String DP Framework

Most string DP problems follow this pattern:

**Two strings:** `dp[i][j]` = answer for `s1[0:i]` and `s2[0:j]`

**One string:** `dp[i][j]` = answer for substring `s[i:j+1]`

The key question at each cell: **do the characters match?**
- Match → use diagonal `dp[i-1][j-1]` (+ something)
- No match → use `dp[i-1][j]`, `dp[i][j-1]`, or `dp[i-1][j-1]` (+ cost)

---

## 🎯 Problem 1: Longest Common Subsequence (LC #1143)

Find the length of the longest subsequence common to both strings.

```
s1 = "abcde", s2 = "ace" → LCS = "ace" → length 3
```

**State:** `dp[i][j]` = LCS length of `s1[0:i]` and `s2[0:j]`

**Transition:**
- Characters match (`s1[i-1] == s2[j-1]`): extend LCS → `dp[i-1][j-1] + 1`
- Don't match: best of skipping either char → `max(dp[i-1][j], dp[i][j-1])`

In [ ]:
def longestCommonSubsequence(text1, text2):
    """
    Classic 2D string DP.
    Time: O(m*n), Space: O(n) with optimization
    """
    m, n = len(text1), len(text2)
    dp = [0] * (n + 1)
    
    for i in range(1, m + 1):
        prev = 0  # dp[i-1][j-1]
        for j in range(1, n + 1):
            temp = dp[j]  # Save dp[i-1][j] before overwrite
            if text1[i-1] == text2[j-1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j-1])  # max(above, left)
            prev = temp
    
    return dp[n]

print(longestCommonSubsequence("abcde", "ace"))    # 3
print(longestCommonSubsequence("abc", "def"))      # 0

**LCS is a building block.** Many problems reduce to LCS:
- Shortest common supersequence = `m + n - LCS`
- Minimum deletions to make strings equal = `m + n - 2*LCS`

---

## 🎯 Problem 2: Edit Distance (LC #72)

Minimum operations (insert, delete, replace) to convert `word1` → `word2`.

```
"horse" → "ros"
horse → rorse (replace h→r)
rorse → rose (delete r)
rose → ros (delete e)  → 3 operations
```

**Transition:**
- Match: `dp[i-1][j-1]` (no cost)
- No match: `1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])` (replace, delete, insert)

In [ ]:
def minDistance(word1, word2):
    """
    Edit Distance DP.
    Time: O(m*n), Space: O(n)
    """
    m, n = len(word1), len(word2)
    dp = list(range(n + 1))  # Base: converting "" to word2[0:j] costs j insertions
    
    for i in range(1, m + 1):
        prev = dp[0]  # dp[i-1][0]
        dp[0] = i     # Converting word1[0:i] to "" costs i deletions
        for j in range(1, n + 1):
            temp = dp[j]
            if word1[i-1] == word2[j-1]:
                dp[j] = prev  # No operation needed
            else:
                dp[j] = 1 + min(
                    prev,     # Replace
                    dp[j],    # Delete (above = already converted shorter word1)
                    dp[j-1]   # Insert (left = already matched shorter word2)
                )
            prev = temp
    
    return dp[n]

print(minDistance("horse", "ros"))       # 3
print(minDistance("intention", "execution"))  # 5

---

## 🎯 Problem 3: Word Break (LC #139)

Can `s` be segmented into space-separated dictionary words?

**State:** `dp[i]` = True if `s[0:i]` can be segmented.

**Transition:** `dp[i] = True` if there exists some `j < i` where `dp[j]` is True AND `s[j:i]` is in the dictionary.

In [ ]:
def wordBreak(s, wordDict):
    """
    1D DP: can s[0:i] be segmented?
    Time: O(n² * k) where k = avg word length, Space: O(n)
    """
    word_set = set(wordDict)
    dp = [False] * (len(s) + 1)
    dp[0] = True  # Empty string is always valid
    
    for i in range(1, len(s) + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break  # Found valid segmentation, no need to check more
    
    return dp[len(s)]

print(wordBreak("leetcode", ["leet", "code"]))          # True
print(wordBreak("applepenapple", ["apple", "pen"]))     # True
print(wordBreak("catsandog", ["cats","dog","sand","and","cat"]))  # False

---

## 🎯 Problem 4: Longest Palindromic Subsequence (LC #516)

Find the length of the longest palindromic subsequence in a string.

**Clever reduction:** LPS(s) = LCS(s, reverse(s))!

Or directly: `dp[i][j]` = LPS of `s[i:j+1]`.
- If `s[i] == s[j]`: `dp[i][j] = dp[i+1][j-1] + 2`
- Else: `dp[i][j] = max(dp[i+1][j], dp[i][j-1])`

In [ ]:
def longestPalindromeSubseq(s):
    """
    LPS = LCS(s, reverse(s)).
    Time: O(n²), Space: O(n)
    """
    return longestCommonSubsequence(s, s[::-1])

print(longestPalindromeSubseq("bbbab"))   # 4 ("bbbb")
print(longestPalindromeSubseq("cbbd"))    # 2 ("bb")

---

## 🗺️ String DP Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  STRING DP PATTERNS                                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  TWO STRINGS → dp[i][j] for s1[0:i], s2[0:j]:                    ║
║    LCS: match → diag+1, else → max(above, left)                   ║
║    Edit Dist: match → diag, else → 1+min(diag, above, left)       ║
║                                                                      ║
║  ONE STRING → dp[i][j] for s[i..j]:                               ║
║    Palindrome: ends match → dp[i+1][j-1]+2                        ║
║    Fill diagonally or bottom-up (i from n-1 to 0)                  ║
║                                                                      ║
║  SEGMENTATION → dp[i] = can s[0:i] be split?                      ║
║    Word Break: try all split points j                              ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 1143 | Longest Common Subsequence | Medium | Classic 2-string DP |
| 72 | Edit Distance | Hard | 3 operations: insert/delete/replace |
| 139 | Word Break | Medium | 1D DP with set lookup |
| 516 | Longest Palindromic Subsequence | Medium | LCS(s, reverse(s)) |
| 5 | Longest Palindromic Substring | Medium | Expand from center or DP |
| 97 | Interleaving String | Medium | dp[i][j] = can form interleave? |
| 115 | Distinct Subsequences | Hard | Count ways s1 contains s2 |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Two strings? → dp[i][j]. Check if chars match at (i,j).      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 LCS is a building block for many string problems               │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Edit Distance: match=free, else 1+min(replace, delete, insert)│
├──────────────────────────────────────────────────────────────────────┤
│  📌 Palindrome subsequence = LCS(s, reverse(s)). Elegant!         │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Knapsack Patterns

The famous 0/1 knapsack and its many disguises.